# Annotate Generated Images

Workflow for pseudo-labeling and correcting annotations on AI-generated goban images.

**Sections:**
1. Run predictions — pseudo-label generated images with the current model
2. Launch annotator — open the annotation UI to review/correct predictions
3. Review corrections — inspect annotation stats

> This notebook uses a **separate** workspace (`data/annotate_generated/`) so
> existing v2 corrections in `data/annotate/corrected.json` are never touched.

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
IMAGES_DIR = Path("../data/generated")  # raw generated images
ANNOTATE_DIR = Path("../data/annotate_generated")  # annotator workspace (separate from v2)
CORRECTED_FILE = ANNOTATE_DIR / "corrected.json"
SERVER_PORT = 7861  # different port from v2 annotator

MODEL_NAME = "kaya-go/moku-v2"  # model for pseudo-labels
THRESHOLD = 0.01  # very low → use the Score slider in the annotator to filter
BATCH_SIZE = 4

ANNOTATE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Images dir:  {IMAGES_DIR.resolve()}")
print(f"Workspace:   {ANNOTATE_DIR.resolve()}")

Images dir:  /Users/hadim/Code/libs/moku/data/generated
Workspace:   /Users/hadim/Code/libs/moku/data/annotate_generated


## 1. Run Predictions (Pseudo-labeling)

Loads the current best model and runs inference on all generated images.
Outputs `images.json` + copies images into the annotator workspace.

Alternatively, run from the CLI:
```bash
pixi run moku predict --images-dir data/generated --out-dir data/annotate_generated --threshold 0.3
```

In [2]:
import shutil

import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import RTDetrForObjectDetection

from moku.model import load_image_processor

# Gather all images
all_image_paths = sorted(
    p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
)

# Prepare output dir
out_images_dir = ANNOTATE_DIR / "images"
out_images_dir.mkdir(parents=True, exist_ok=True)

# Resume: load existing images.json if present
images_json_path = ANNOTATE_DIR / "images.json"
images_meta = []
annotations = {}
already_done = set()

if images_json_path.exists():
    with open(images_json_path) as f:
        existing_data = json.load(f)
    images_meta = existing_data.get("images", [])
    annotations = existing_data.get("annotations", {})
    already_done = {img["filename"] for img in images_meta}

# Filter to only new images
image_paths = [p for p in all_image_paths if p.name not in already_done]
print(f"Found {len(all_image_paths)} images total, {len(already_done)} already predicted, {len(image_paths)} new")

if not image_paths:
    print("Nothing to do — all images already predicted.")
else:
    # Load model only if there's work to do
    image_processor = load_image_processor(MODEL_NAME)
    model = RTDetrForObjectDetection.from_pretrained(MODEL_NAME)

    device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Model loaded on {device}")

    with torch.no_grad():
        for batch_start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Predicting"):
            batch_paths = image_paths[batch_start : batch_start + BATCH_SIZE]
            pil_images = [Image.open(p).convert("RGB") for p in batch_paths]

            inputs = image_processor(images=pil_images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)

            target_sizes = torch.tensor([[img.height, img.width] for img in pil_images]).to(device)
            results = image_processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=THRESHOLD
            )

            for path, img, result in zip(batch_paths, pil_images, results):
                filename = path.name

                # Copy image
                dst = out_images_dir / filename
                if not dst.exists():
                    shutil.copy2(path, dst)

                images_meta.append({
                    "id": filename,
                    "filename": filename,
                    "width": img.width,
                    "height": img.height,
                    "source": "generated",
                })

                boxes_list = []
                for ann_id, (box, score, label) in enumerate(
                    zip(
                        result["boxes"].cpu().tolist(),
                        result["scores"].cpu().tolist(),
                        result["labels"].cpu().tolist(),
                    )
                ):
                    x1, y1, x2, y2 = box
                    boxes_list.append({
                        "id": ann_id,
                        "x": round(x1, 2),
                        "y": round(y1, 2),
                        "w": round(x2 - x1, 2),
                        "h": round(y2 - y1, 2),
                        "category": int(label),
                        "score": round(score, 4),
                    })

                annotations[filename] = {"boxes": boxes_list}

    # Write images.json (merged: existing + new)
    with open(images_json_path, "w") as f:
        json.dump({"images": images_meta, "annotations": annotations}, f, indent=2)

    n_new = len(image_paths)
    n_total = sum(len(a["boxes"]) for a in annotations.values())
    print(f"\nDone — {n_new} new images predicted, {len(images_meta)} total, {n_total} boxes")
    print(f"Workspace ready at {ANNOTATE_DIR}")

Found 500 images total, 500 already predicted, 0 new
Nothing to do — all images already predicted.


## 2. Launch Annotator

Opens the annotation UI on a **separate port** so it doesn't conflict with the v2 annotator.

**Tools:** `1` Corner · `2` Black stone · `3` White stone · `V` Move/Pan  
**Zoom:** Scroll wheel · `0` Reset fit · `Space+Drag` Pan  
**Edit:** Click to place · Drag to move · Right-click to delete · `Del` delete selected  
**Navigate:** `←`/`→` Prev/Next · `Shift+←`/`→` Jump to next flagged  
**Other:** `Ctrl+S` Save · `M` Toggle magnifier · Filter dropdown in sidebar  

> ⚠ Run the cell below and open the printed URL. Press the interrupt button (■) to stop.

In [3]:
import subprocess
import webbrowser

PROJECT_ROOT = Path.cwd().parent
SERVER_SCRIPT = PROJECT_ROOT / "tools" / "annotator" / "server.py"

server_proc = subprocess.Popen(
    ["python", str(SERVER_SCRIPT),
     "--data-dir", str(ANNOTATE_DIR.resolve()),
     "--output", str(CORRECTED_FILE.resolve()),
     "--port", str(SERVER_PORT)],
)

url = f"http://localhost:{SERVER_PORT}"
print(f"Annotator running at {url}")
print("Press the interrupt button (■) to stop the server.")
webbrowser.open(url)

try:
    server_proc.wait()
except KeyboardInterrupt:
    server_proc.terminate()
    print("\nServer stopped.")

Annotator running at http://localhost:7861
Press the interrupt button (■) to stop the server.
Moku Annotator  →  http://localhost:7861
  data-dir  : /Users/hadim/Code/libs/moku/data/annotate_generated
  output    : /Users/hadim/Code/libs/moku/data/annotate_generated/corrected.json
Press Ctrl+C to stop.

Stopped.

Server stopped.


## 3. Review Corrections

Inspect stats on the corrected annotations.

In [4]:
import pandas as pd
from IPython.display import display

CATEGORY_NAMES = {0: "black_stone", 1: "white_stone", 2: "board_corner"}

if not CORRECTED_FILE.exists():
    print(f"No corrections file found at {CORRECTED_FILE}. Run the annotator first.")
else:
    with open(CORRECTED_FILE) as f:
        corrections = json.load(f)

    print(f"Corrections for {len(corrections)} / {len(images_meta)} images")

    # Category breakdown
    cat_counts = {name: 0 for name in CATEGORY_NAMES.values()}
    for fname, corr in corrections.items():
        for box in corr.get("boxes", []):
            cat_name = CATEGORY_NAMES.get(box["category"], "unknown")
            cat_counts[cat_name] = cat_counts.get(cat_name, 0) + 1

    print("\nCategory counts (corrected images):")
    display(pd.Series(cat_counts, name="count").to_frame())

    # Per-image stats
    rows = []
    for fname, corr in corrections.items():
        boxes = corr.get("boxes", [])
        n_corners = sum(1 for b in boxes if b["category"] == 2)
        n_black = sum(1 for b in boxes if b["category"] == 0)
        n_white = sum(1 for b in boxes if b["category"] == 1)
        rows.append({"filename": fname, "corners": n_corners, "black": n_black, "white": n_white})

    df = pd.DataFrame(rows)
    print(f"\nImages with != 4 corners: {(df['corners'] != 4).sum()}")
    display(df.describe())

Corrections for 112 / 500 images

Category counts (corrected images):


,count
black_stone,4632
white_stone,4910
board_corner,448



Images with != 4 corners: 0


,corners,black,white
count,112.0,112.000000,112.000000
mean,4.0,41.357143,43.839286
std,0.0,35.028394,39.379089
min,4.0,4.000000,3.000000
25%,4.0,11.000000,11.000000
50%,4.0,28.500000,30.000000
75%,4.0,65.000000,66.000000
max,4.0,127.000000,161.000000
